In [5]:
import pandas as pd
import numpy as np
import os

# Load the raw CDC YRBS 2007 dataset
file_name = "YRBS_2007 (1).csv"
df = pd.read_csv(file_name)

print("==================================================================")
print("             DATA CHECK STEP 1: INITIAL SUMMARY                   ")
print("==================================================================")
print(f"✅ Raw Dataset successfully loaded.")
print(f"🔹 Total Number of Rows (Students Surveyed): {df.shape[0]}")
print(f"🔹 Total Number of Columns (Variables): {df.shape[1]}")
print("==================================================================")

# Display a quick preview of the raw columns
df.head(3)

             DATA CHECK STEP 1: INITIAL SUMMARY                   
✅ Raw Dataset successfully loaded.
🔹 Total Number of Rows (Students Surveyed): 14041
🔹 Total Number of Columns (Variables): 103


,RaceEth,HowOldAreYou,WhatIsYourSex,InWhatGradeAreYou,AreYouHispanicOrLatino,WhatIsYourRace,HowTallAreYouWithoutShoesInMeters,HowMuchDoYouWeighWithoutShoesInKG,BicyleHelmetUse,SeatBeltUse,...,InjuredWhileExercising,HIVTesting,SunscreenUse,SunProtection,Sleep,HealthInGeneral,BMIPCT,weight,stratum,psu
0,7.0,4.0,2.0,2.0,1.0,C,NaN,NaN,2.0,1.0,...,3.0,2.0,1.0,1.0,5.0,3.0,NaN,1.5104,101,11030
1,5.0,7.0,2.0,2.0,2.0,E,1.7,68.04,4.0,4.0,...,2.0,3.0,1.0,5.0,4.0,3.0,66.531824,1.8559,101,11030
2,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,5.0,3.0,...,2.0,3.0,2.0,1.0,1.0,1.0,NaN,1.8559,101,11030


In [6]:
print("==================================================================")
print("      DATA CHECK STEP 2: SEARCHING FOR SEXUAL ORIENTATION         ")
print("==================================================================")
print("Hypothesis: Investigating if specific 'SexualOrientation' or 'LGB'")
print("identity variables exist in this standard YRBS 2007 dataset.\n")

# Search for any columns containing 'sex' or 'orient' keyword (case-insensitive)
potential_cols = [col for col in df.columns if 'sex' in col.lower() or 'orient' in col.lower()]

for i, col in enumerate(potential_cols):
    print(f"📍 Potential Match {i+1}: '{col}'")

print("\n[Methodological Conclusion]:")
print("The search confirms that an explicit 'SexualOrientation' variable is NOT available.")
print("The columns with 'sex' are restricted to biological sex ('WhatIsYourSex') or sexual behaviors.")
print("Therefore, to maintain statistical rigor, the study pivots to Gender-based Marginalization.")

      DATA CHECK STEP 2: SEARCHING FOR SEXUAL ORIENTATION         
Hypothesis: Investigating if specific 'SexualOrientation' or 'LGB'
identity variables exist in this standard YRBS 2007 dataset.

📍 Potential Match 1: 'WhatIsYourSex'
📍 Potential Match 2: 'ForcedSexualIntercourse'
📍 Potential Match 3: 'EverSexualIntercourse'
📍 Potential Match 4: 'FirstSexualIntercourse'
📍 Potential Match 5: 'MultipleSexPartners'
📍 Potential Match 6: 'CurrentSexualActivity'
📍 Potential Match 7: 'AlcoholDrugsAndSex'

[Methodological Conclusion]:
The search confirms that an explicit 'SexualOrientation' variable is NOT available.
The columns with 'sex' are restricted to biological sex ('WhatIsYourSex') or sexual behaviors.
Therefore, to maintain statistical rigor, the study pivots to Gender-based Marginalization.


In [7]:
print("==================================================================")
print("      DATA CHECK STEP 3: MISSING VALUE & QUALITY ANALYSIS         ")
print("==================================================================")

# Define our 4 core columns for the intersectional framework
analysis_cols = [
    'RaceEth', 
    'WhatIsYourSex', 
    'SafetyConcernsAtSchool', 
    'WereThreatenedOrInjuredWithAWeaponOnSchoolProperty'
]

# Calculate missing data before deletion
print("--- Missing Data Summary Per Core Variable ---")
for col in analysis_cols:
    missing_count = df[col].isna().sum()
    missing_pct = (missing_count / df.shape[0]) * 100
    print(f"Variable '{col}': Missing {missing_count} rows ({missing_pct:.2f}%)")

# Create the subset and perform Listwise Deletion
sub_df = df[analysis_cols].copy()
initial_rows = sub_df.shape[0]
sub_df.dropna(subset=analysis_cols, inplace=True)
final_rows = sub_df.shape[0]

print("\n--- Listwise Deletion Impact ---")
print(f"Initial Row Count: {initial_rows}")
print(f"Cleaned Row Count (Valid Samples): {final_rows}")
print(f"Total Rows Dropped due to Missingness: {initial_rows - final_rows}")
print(f"Data Retention Rate: {(final_rows / initial_rows)*100:.2f}%")
print("==================================================================")

      DATA CHECK STEP 3: MISSING VALUE & QUALITY ANALYSIS         
--- Missing Data Summary Per Core Variable ---
Variable 'RaceEth': Missing 248 rows (1.77%)
Variable 'WhatIsYourSex': Missing 13 rows (0.09%)
Variable 'SafetyConcernsAtSchool': Missing 147 rows (1.05%)
Variable 'WereThreatenedOrInjuredWithAWeaponOnSchoolProperty': Missing 147 rows (1.05%)

--- Listwise Deletion Impact ---
Initial Row Count: 14041
Cleaned Row Count (Valid Samples): 13645
Total Rows Dropped due to Missingness: 396
Data Retention Rate: 97.18%


In [8]:
print("==================================================================")
print("          DATA CHECK STEP 4: RECODING & DATA EXPORT               ")
print("==================================================================")

# 1. Dummy Recoding for Independent Variables (0 = Baseline Majority, 1 = Target Minority)
# RaceEth: Code 5.0 is non-Hispanic White. All others are mapped to 1 (Minority)
sub_df['Is_Minority_Race'] = np.where(sub_df['RaceEth'] == 5.0, 0, 1)

# WhatIsYourSex: Code 1.0 is Male (Baseline), Code 2.0 is Female (Target)
sub_df['Is_Female'] = np.where(sub_df['WhatIsYourSex'] == 1.0, 0, 1)

# 2. Standardizing Dependent Variables to have a True Zero Origin
# Subtract 1 so that '0' perfectly means '0 days/times' for regression interpretation clarity
sub_df['School_Safety_Index'] = sub_df['SafetyConcernsAtSchool'] - 1
sub_df['Weapon_Threat_Index'] = sub_df['WereThreatenedOrInjuredWithAWeaponOnSchoolProperty'] - 1

# 3. Create the directories automatically based on GitHub requirements
output_dir = "data/processed/"
output_file = os.path.join(output_dir, "cleaned_data.csv")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"📁 Directory '{output_dir}' successfully created!")

# 4. Save the finalized cleaned dataset
sub_df.to_csv(output_file, index=False)

print("\n🎉 SUCCESS! CLEANED DATASET GENERATED.")
print(f"💾 File saved at: {output_file}")
print(f"📊 Final columns exported: {list(sub_df.columns)}")
print("==================================================================")

# Show the rich preview of the final dataset
sub_df.head()

          DATA CHECK STEP 4: RECODING & DATA EXPORT               

🎉 SUCCESS! CLEANED DATASET GENERATED.
💾 File saved at: data/processed/cleaned_data.csv
📊 Final columns exported: ['RaceEth', 'WhatIsYourSex', 'SafetyConcernsAtSchool', 'WereThreatenedOrInjuredWithAWeaponOnSchoolProperty', 'Is_Minority_Race', 'Is_Female', 'School_Safety_Index', 'Weapon_Threat_Index']


,RaceEth,WhatIsYourSex,SafetyConcernsAtSchool,WereThreatenedOrInjuredWithAWeaponOnSchoolProperty,Is_Minority_Race,Is_Female,School_Safety_Index,Weapon_Threat_Index
0,7.0,2.0,5.0,6.0,1,1,4.0,5.0
1,5.0,2.0,1.0,1.0,0,1,0.0,0.0
3,7.0,1.0,1.0,1.0,1,0,0.0,0.0
4,7.0,1.0,1.0,1.0,1,0,0.0,0.0
5,7.0,1.0,1.0,1.0,1,0,0.0,0.0
